In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress
import matplotlib as mpl

## Functions 

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
def make_test_map(data,hdr,vmin=-0.5,vmax=0.5,cmap='RdBu_r',
                  *args,**kwargs):

    wcs = WCS(hdr)
    fs = 20
    plt.figure(figsize=(40,4))
    plt.subplot(projection=wcs)
    plt.imshow(data,origin='lower',vmin=vmin,vmax=vmax,cmap=cmap)
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    plt.title(hdr['OBJECT'],fontsize=fs)
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs)

    gc.collect()

    return

In [ ]:
def make_zoomed_map(data,hdr,vmin=-0.5,vmax=0.5,cmap='RdBu_r',
                    llim = [192,52], blim = [-7,10],
                    *args,**kwargs):
    
    aspect = (blim[1]-blim[0])/(llim[0]-llim[1])
    #print(aspect)
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    
    fig = plt.figure(figsize=(20,20*aspect*0.8))
    ax  = fig.add_subplot(111, projection=WCS(hdr).celestial)
    im  = ax.imshow(data, origin='lower', vmin=vmin, vmax=vmax,cmap=cmap)
    ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    cbar = fig.colorbar(im)
    cbar.set_label('')

    return

In [ ]:
def make_landecker_map(data1,data2,hdr,v1max=300,v2max=1,
                       cmap1='RdBu_r',cmap2 = 'viridis',
                       llim = [192,52], blim = [-7,10],filename='test',
                       *args,**kwargs):
    
    aspect = (blim[1]-blim[0])/(llim[0]-llim[1])
    print(aspect)
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 22
    
    fig = plt.figure(figsize=(16,11.3))
    
    plt.subplots_adjust(hspace=0.0,left=0.08, right=0.98, top=0.99, bottom=0.08)
    
    cmap = mpl.colormaps.get_cmap(cmap1)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
    
    ax1  = fig.add_subplot(211, projection=WCS(hdr).celestial)
    im1  = ax1.imshow(data1, origin='lower', vmin=-v1max, vmax=v1max,cmap=cmap)
    ax1.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax1.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax1.set_xticks([125,130,135])
    cbar1 = fig.colorbar(im1, ax=ax1, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar1.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar1.set_ticks([-200,-100,0,100,200])

    cmap = mpl.colormaps.get_cmap(cmap2)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
   
    ax2  = fig.add_subplot(212, projection=WCS(hdr).celestial)
    im2  = ax2.imshow(data2, origin='lower', vmin=0, vmax=v2max,cmap=cmap)
    ax2.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax2.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax2.set_xticks([125,130,135])
    cbar2 = fig.colorbar(im2, ax=ax2, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar2.set_label(r'PI (K)', fontsize=fs)
    cbar2.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
    
    
    ax2.set_xlabel('Galactic Longitude',fontsize=fs)
    fig.text(0.02,0.45,'Galactic Latitude',fontsize=fs,rotation='vertical')
    for ax in [ax1,ax2]:
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        
    for cbar in [cbar1,cbar2]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
    
    #plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.pdf')
    #plt.savefig('/home/aordog/Python/cgps-gmims/'+filename+'.pdf')

    return

## Test different ways of masking

### Read in data and make test plots

In [ ]:
# Read in RM, Pearson R, and standard error in RM:
#hdu_RM_CG = fits.open('/srv/aordog/cgps_gmims_data/RM_CG_conv4_regrd.fits')
hdu_RM_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_CG_conv4_regrd.fits')
RM_CG_all = hdu_RM_CG[0].data
RM_CG     = RM_CG_all.copy()
rvalue_CG = hdu_RM_CG[2].data
stderr_CG = hdu_RM_CG[4].data
hdr       = hdu_RM_CG[0].header

hdu_RM_G = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_G_regrd.fits')
RM_G_all = hdu_RM_G[0].data
RM_G     = RM_G_all.copy()
rvalue_G = hdu_RM_G[2].data
stderr_G = hdu_RM_G[4].data

hdu_RM_C = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_C_conv4_regrd.fits')
RM_C_all = hdu_RM_C[0].data
RM_C     = RM_C_all.copy()
rvalue_C = hdu_RM_C[2].data
stderr_C = hdu_RM_C[4].data


# Read in polarised intensity:
#hdu_PI_CG = fits.open('/srv/aordog/cgps_gmims_data/PI_CG_conv4_regrd_avg_PI.fits')
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_CG_conv4_regrd_PI_of_mean.fits')
PI_CG     = hdu_PI_CG[0].data

hdu_PI_G = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_G_regrd_PI_of_mean.fits')
PI_G     = hdu_PI_G[0].data 

hdu_PI_C = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_C_conv4_regrd_PI_of_mean.fits')
PI_C     = hdu_PI_C[0].data 


# Set outside of mosaics to NaN:
RM_CG[RM_CG_all==0.0]     = np.nan
rvalue_CG[RM_CG_all==0.0] = np.nan
stderr_CG[RM_CG_all==0.0] = np.nan
PI_CG[RM_CG_all==0.0]     = np.nan

RM_C[RM_C_all==0.0]     = np.nan
rvalue_C[RM_C_all==0.0] = np.nan
stderr_C[RM_C_all==0.0] = np.nan
PI_C[RM_C_all==0.0]     = np.nan

make_zoomed_map(RM_CG,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [192,52], blim = [-4,6])
make_zoomed_map(stderr_CG,hdr,vmin=0,vmax=200,cmap='viridis',llim = [192,52], blim = [-4,6])
make_zoomed_map(PI_CG,hdr,vmin=0,vmax=1,cmap='cubehelix',llim = [192,52], blim = [-4,6])


## Read in Q, U noise, and average Q, U channels for S:N estimate

In [ ]:
Q_noise, hdrQ_noise = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','CG_conv4_regrd_noise')
U_noise, hdrU_noise = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','CG_conv4_regrd_noise')
#Q_noise, hdrQ_noise = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','C_conv4_regrd_noise')
#U_noise, hdrU_noise = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','C_conv4_regrd_noise')

Q_list, hdrQ_list = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','CG_conv4_regrd')
U_list, hdrU_list = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','CG_conv4_regrd')
#Q_list, hdrQ_list = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','C_conv4_regrd')
#U_list, hdrU_list = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','C_conv4_regrd')

Qmean = (1/4)*(Q_list[0]+Q_list[1]+Q_list[2]+Q_list[3])
Umean = (1/4)*(U_list[0]+U_list[1]+U_list[2]+U_list[3])


## Noise calculation

In [ ]:
dQ = (1/4)*np.sqrt(Q_noise[0]**2 + Q_noise[1]**2 + Q_noise[2]**2 + Q_noise[3]**2)
dU = (1/4)*np.sqrt(U_noise[0]**2 + U_noise[1]**2 + U_noise[2]**2 + U_noise[3]**2)

dPI = np.sqrt((Qmean**2)*(dQ**2) + (Umean**2)*(dU**2))/PI_CG
#dPI = np.sqrt((Qmean**2)*(dQ**2) + (Umean**2)*(dU**2))/PI_C

plt.imshow(dPI,vmin=0,vmax=0.1)

In [ ]:
Qpeak=plt.hist(Q_noise[0].flatten(),bins=100,range=(0,0.12),label='one channel Q noise',alpha=0.6);
Upeak=plt.hist(U_noise[0].flatten(),bins=100,range=(0,0.12),label='one channel U noise',alpha=0.6);
PIpeak=plt.hist(dPI.flatten(),bins=100,range=(0,0.12),alpha=0.6,label='PI noise');

idx = np.where(Qpeak[0] == max(Qpeak[0]))
print(Qpeak[1][idx])
plt.axvline(Qpeak[1][idx],color='C0')

idx = np.where(Upeak[0] == max(Upeak[0]))
print(Upeak[1][idx])
plt.axvline(Upeak[1][idx],color='C1')

idx = np.where(PIpeak[0] == max(PIpeak[0]))
print(PIpeak[1][idx])
plt.axvline(PIpeak[1][idx],color='C2')

plt.grid()
plt.legend()

plt.savefig('../plots/noise_hist_CGPS_GMIMS.pdf')

### Histograms of everything

In [ ]:
fig = plt.figure(figsize=(16,7))

ax1 = fig.add_subplot(211)
ax1.hist(RM_CG.flatten(),bins=1001,range=(-500,500),color='C0');
ax1.hist(stderr_CG.flatten(),bins=501,range=(0,500),alpha=0.7,color='C1');
ax1.set_xlim(-300,300)
ax1.set_xlabel(r'RM and $\sigma_{RM}$')
ax1.grid()

ax2 = fig.add_subplot(212)
ax2.hist(rvalue_CG.flatten(),bins=501,range=(-1,1),color='C0');
ax2.set_xlim(-1,1)
ax2.set_xlabel('r-value')
ax2.grid()

In [ ]:
fig = plt.figure(figsize=(18,5))

ax1 = fig.add_subplot(141)
ax1.hist2d(RM_CG[np.isfinite(RM_CG)].flatten(),stderr_CG[np.isfinite(RM_CG)].flatten(),
           range=([[-500,500],[0,500]]),bins=(1001,501), cmap='cubehelix',vmin=0,vmax=500);
ax1.set_xlim(-200,200)
ax1.set_ylim(0,200)
ax1.set_xlabel('RM')
ax1.set_ylabel(r'$\sigma_{RM}$')

ax2 = fig.add_subplot(142)
ax2.hist2d(RM_CG[np.isfinite(RM_CG)].flatten(),stderr_CG[np.isfinite(RM_CG)].flatten()/RM_CG[np.isfinite(RM_CG)].flatten(),
           range=([[-500,500],[-20,20]]),bins=(1001,1001), cmap='cubehelix',vmin=0,vmax=500);
ax2.set_xlim(-200,200)
ax2.set_ylim(-20,20)
ax2.set_xlabel('RM')
ax2.set_ylabel(r'$\sigma_{RM}/RM$')

ax3 = fig.add_subplot(143)
ax3.hist2d(RM_CG[np.isfinite(RM_CG)].flatten(),rvalue_CG[np.isfinite(RM_CG)].flatten(),
           range=([[-500,500],[-2,2]]),bins=(1001,1001), cmap='cubehelix',vmin=0,vmax=500);
ax3.set_xlim(-200,200)
ax3.set_ylim(-1.1,1.1)
ax3.set_xlabel('RM')
ax3.set_ylabel('r-value')

ax4 = fig.add_subplot(144)
ax4.hist2d(PI_CG[np.isfinite(RM_CG)].flatten(),stderr_CG[np.isfinite(RM_CG)].flatten(),
           range=([[0,3],[0,500]]),bins=(1001,1001), cmap='cubehelix',vmin=0,vmax=500);
ax4.set_xlim(0,1)
ax4.set_ylim(0,200)
ax4.set_xlabel('PI')
ax4.set_ylabel(r'$\sigma_{RM}$')
ax4.grid()

### Mask low r-value pixels
This tends to mask out low-RM values, as the r is small when the slope is small

In [ ]:
RM_CG_filt = RM_CG.copy()
########################################
RM_CG_filt[abs(rvalue_CG)<0.3] = np.nan
########################################

fig = plt.figure(figsize=(16,3))
ax1 = fig.add_subplot(111)
ax1.hist(RM_CG.flatten(),bins=1001,range=(-500,500),color='C0');
ax1.hist(RM_CG_filt.flatten(),bins=1001,range=(-500,500),color='C1',alpha=0.7);
ax1.set_xlim(-300,300)
ax1.set_xlabel('RM')
ax1.grid()

make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [170,130], blim = [-4,6])
make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [100,60], blim = [-4,6])

### Mask high sigma_RM pixels
This tends to mask out high RM values, but it should probably be implemented since there is no value in high uncertainty RMs. However, setting a PI threshold is probably similar (see 2D histogram of sigma vs PI above)

In [ ]:
RM_CG_filt = RM_CG.copy()
#########################################
RM_CG_filt[abs(stderr_CG)>100] = np.nan
#########################################

fig = plt.figure(figsize=(16,3))
ax1 = fig.add_subplot(111)
ax1.hist(RM_CG.flatten(),bins=1001,range=(-500,500),color='C0');
ax1.hist(RM_CG_filt.flatten(),bins=1001,range=(-500,500),color='C1',alpha=0.7);
ax1.set_xlim(-300,300)
ax1.set_xlabel('RM')
ax1.grid()

make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [170,130], blim = [-4,6])
make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [100,60], blim = [-4,6])

### Mask low PI pixels
This is similar to cutting out high sigma (>100) but keeps a little bit more in Cygnus X region and around artifacts (which are high in PI but not actually trustworthy)

In [ ]:
RM_CG_filt = RM_CG.copy()
##################################
RM_CG_filt[PI_CG < 0.1] = np.nan
##################################

fig = plt.figure(figsize=(16,3))
ax1 = fig.add_subplot(111)
ax1.hist(RM_CG.flatten(),bins=1001,range=(-500,500),color='C0');
ax1.hist(RM_CG_filt.flatten(),bins=1001,range=(-500,500),color='C1',alpha=0.7);
ax1.set_xlim(-300,300)
ax1.set_xlabel('RM')
ax1.grid()

make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [170,130], blim = [-4,6])
make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [100,60], blim = [-4,6])

### Mask low PI pixels and high sigma_RM pixels
Masking out low PI is well motivated, so we should include this, and high uncertainty RMs also make sense to include.

In [ ]:
RM_CG_filt = RM_CG.copy()
########################################
RM_CG_filt[PI_CG<0.1] = np.nan
RM_CG_filt[stderr_CG>100] = np.nan
########################################

fig = plt.figure(figsize=(16,3))
ax1 = fig.add_subplot(111)
ax1.hist(RM_CG.flatten(),bins=1001,range=(-500,500),color='C0');
ax1.hist(RM_CG_filt.flatten(),bins=1001,range=(-500,500),color='C1',alpha=0.7);
ax1.set_xlim(-300,300)
ax1.set_xlabel('RM')
ax1.grid()

make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [170,130], blim = [-4,6])
make_zoomed_map(RM_CG_filt,hdr,vmin=-200,vmax=200,cmap='Spectral_r',llim = [100,60], blim = [-4,6])

## Making figures for the paper

In [ ]:
RM_CG_filt = RM_CG.copy()
RM_G_filt = RM_G.copy()
RM_C_filt = RM_C.copy()
########################################
RM_CG_filt[PI_CG<0.1] = np.nan
RM_CG_filt[stderr_CG>100] = np.nan
RM_G_filt[PI_G<0.1] = np.nan
RM_G_filt[stderr_G>100] = np.nan
#RM_C_filt[PI_C<0.1] = np.nan
RM_C_filt[stderr_C>100] = np.nan
########################################

llim_list = [[86,66],[104,84],[122,102],[140,120],[158,138],[176,156],[194,174]]
name_list = ['66_86','84_104','102_122','120_140','138_158','156_176','174_194']

#make_landecker_map(RM_CG_filt,PI_CG,hdr,v1max=300,v2max=0.6,
#                   cmap1='RdBu_r',cmap2='gist_heat_r',
#                    llim = [86,52], blim = [-6,7.6],filename = 'CGPS_GMIMS_RM_52_86_grey')

#make_landecker_map(RM_G_filt,PI_G,hdr,v1max=300,v2max=0.6,
#                   cmap1='RdBu_r',cmap2='gist_heat_r',
#                    llim = [86,52], blim = [-6,7.6],filename = 'GMIMS_RM_52_86_grey')

make_landecker_map(RM_C_filt,PI_C,hdr,v1max=300,v2max=0.6,
                   cmap1='RdBu_r',cmap2='gist_heat_r',
                    llim = [86,52], blim = [-6,7.6],filename = 'CGPS_RM_52_86_grey')

for i in range(0,7):
    #make_landecker_map(RM_CG_filt,PI_CG,hdr,v1max=300,v2max=0.6,
    #                   cmap1='RdBu_r',cmap2='gist_heat_r',
    #                    llim = llim_list[i], blim = [-3,5],filename = 'CGPS_GMIMS_RM_'+name_list[i]+'_grey')

    #make_landecker_map(RM_G_filt,PI_G,hdr,v1max=300,v2max=0.6,
    #                   cmap1='RdBu_r',cmap2='gist_heat_r',
    #                    llim = llim_list[i], blim = [-3,5],filename = 'GMIMS_RM_'+name_list[i]+'_grey')

    make_landecker_map(RM_C_filt,PI_C,hdr,v1max=300,v2max=0.6,
                       cmap1='RdBu_r',cmap2='gist_heat_r',
                        llim = llim_list[i], blim = [-3,5],filename = 'CGPS_RM_'+name_list[i]+'_grey')


## Look at linear fits

In [ ]:
hdu_RM_CG = fits.open('/srv/aordog/cgps_gmims_data/RM_CG_conv4_regrd.fits')
hdu_RM_G  = fits.open('/srv/aordog/cgps_gmims_data/RM_G_conv4_regrd.fits')
RM_CG     = hdu_RM_CG[0].data
RM_G      = hdu_RM_G[0].data
RM_CG_hdr = hdu_RM_CG[0].header

RM_CG[RM_CG==0] = np.nan
RM_G[RM_G==0] = np.nan

rvalue_CG = hdu_RM_CG[2].data
rvalue_G  = hdu_RM_G[2].data

PAint_CG  = hdu_RM_CG[1].data
PAint_G   = hdu_RM_G[1].data

hdu_PI_CG = fits.open('/srv/aordog/cgps_gmims_data/PI_CG_conv4_regrd_avg_PI.fits')
hdu_PI_G = fits.open('/srv/aordog/cgps_gmims_data/PI_G_conv4_regrd_avg_PI.fits')
PI_CG     = hdu_PI_CG[0].data
PI_G     = hdu_PI_G[0].data

hdu_PA_A_CG = fits.open('/srv/aordog/cgps_gmims_data/PA_A_CG_conv4_regrd.fits')
hdu_PA_B_CG = fits.open('/srv/aordog/cgps_gmims_data/PA_B_CG_conv4_regrd.fits')
hdu_PA_C_CG = fits.open('/srv/aordog/cgps_gmims_data/PA_C_CG_conv4_regrd.fits')
hdu_PA_D_CG = fits.open('/srv/aordog/cgps_gmims_data/PA_D_CG_conv4_regrd.fits')

hdu_PA_A_G = fits.open('/srv/aordog/cgps_gmims_data/PA_A_G_conv4_regrd.fits')
hdu_PA_B_G = fits.open('/srv/aordog/cgps_gmims_data/PA_B_G_conv4_regrd.fits')
hdu_PA_C_G = fits.open('/srv/aordog/cgps_gmims_data/PA_C_G_conv4_regrd.fits')
hdu_PA_D_G = fits.open('/srv/aordog/cgps_gmims_data/PA_D_G_conv4_regrd.fits')

PA_A_CG = hdu_PA_A_CG[0].data
PA_B_CG = hdu_PA_B_CG[0].data
PA_C_CG = hdu_PA_C_CG[0].data
PA_D_CG = hdu_PA_D_CG[0].data

PA_A_G = hdu_PA_A_G[0].data
PA_B_G = hdu_PA_B_G[0].data
PA_C_G = hdu_PA_C_G[0].data
PA_D_G = hdu_PA_D_G[0].data

In [ ]:
def zoomed_map_linfit(RM1,RM2,PI1,PI2,rvalue1,rvalue2,hdr,PA1,PA2,
                      RMmax=300,PImax=0.5,llim = [192,52], blim = [-7,10],cpt=None,
                     *args,**kwargs):  
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    
    freq = np.array([1406.9,1413.8,1427.4,1434.3])
    lbd2 = ((3e8)/(freq*1e6))**2
    
    lbd2_ext = np.linspace(0.0435,0.0456,100)
    
    fig = plt.figure(figsize=(16,20))
    
    ax1  = fig.add_subplot(421, projection=WCS(hdr).celestial)
    im1  = ax1.imshow(RM1, origin='lower', vmin=-RMmax, vmax=RMmax,cmap='Spectral_r')
    ax1.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax1.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    cbar1 = fig.colorbar(im1)
    cbar1.set_label('')
    
    ax2  = fig.add_subplot(422, projection=WCS(hdr).celestial)
    im2  = ax2.imshow(RM2, origin='lower', vmin=-RMmax, vmax=RMmax,cmap='Spectral_r')
    ax2.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax2.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    cbar2 = fig.colorbar(im2)
    cbar2.set_label('')
    
    ax3  = fig.add_subplot(423, projection=WCS(hdr).celestial)
    im3  = ax3.imshow(PI1, origin='lower', vmin=0, vmax=PImax,cmap='cubehelix')
    ax3.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax3.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    cbar3 = fig.colorbar(im3)
    cbar3.set_label('')
    
    ax4  = fig.add_subplot(424, projection=WCS(hdr).celestial)
    im4  = ax4.imshow(PI2, origin='lower', vmin=0, vmax=PImax,cmap='cubehelix')
    ax4.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax4.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    cbar4 = fig.colorbar(im4)
    cbar4.set_label('')
    
    ax5  = fig.add_subplot(425, projection=WCS(hdr).celestial)
    im5  = ax5.imshow(abs(rvalue1), origin='lower', vmin=0, vmax=1,cmap='viridis')
    ax5.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax5.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    cbar5 = fig.colorbar(im5)
    cbar5.set_label('')
    
    ax6  = fig.add_subplot(426, projection=WCS(hdr).celestial)
    im6  = ax6.imshow(abs(rvalue2), origin='lower', vmin=0, vmax=1,cmap='viridis')
    ax6.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax6.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    cbar6 = fig.colorbar(im6)
    cbar6.set_label('')

    
    if cpt != None:
        for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
            ax.scatter(WCS(hdr).world_to_pixel(cpt)[0],WCS(hdr).world_to_pixel(cpt)[1],
                       color='k',s=20)
        ax7  = fig.add_subplot(427)
        ax7.scatter(lbd2,PA1)
        ax7.set_ylim(-90,90)
        ax7.grid()
    
        ax8  = fig.add_subplot(428)
        ax8.scatter(lbd2,PA2)
        ax8.set_ylim(-90,90)
        ax8.grid()

    return

In [ ]:
cpt = SkyCoord(90.1, 0, frame=Galactic, unit="deg")
#cpt = SkyCoord(90.1, 0.8, frame=Galactic, unit="deg")
idx = WCS(RM_CG_hdr).world_to_pixel(cpt)

ii = int(np.round(idx[0]))
jj = int(np.round(idx[1]))

PA_CG_arr = np.array([PA_A_CG[jj,ii],PA_B_CG[jj,ii],
                      PA_C_CG[jj,ii],PA_D_CG[jj,ii]])*180/np.pi

PA_G_arr = np.array([PA_A_G[jj,ii],PA_B_G[jj,ii],
                     PA_C_G[jj,ii],PA_D_G[jj,ii]])*180/np.pi

print(PA_G_arr)
print(PA_CG_arr)

print(rvalue_G[jj,ii])
print(rvalue_CG[jj,ii])
print('')
print(RM_G[jj,ii])
print(RM_CG[jj,ii])

zoomed_map_linfit(RM_G,RM_CG,PI_G,PI_CG,rvalue_G,rvalue_CG,
                  RM_CG_hdr,PA_G_arr,PA_CG_arr,RMmax=300,PImax=0.5,
                  llim = [92,88], blim = [-2,2],cpt=cpt)

# 